In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

SETS =  [
    "ZZxReto", # Train
    "ZZy1", # Train
    "ZZx2",  # Val
    "ZZy2", # Val
    "LSG-1", # Test
    "LSG-2", # Test
    "ZZx1-inv", # Test
    "ZZx1",  # Test
    "ZZx2-inv", # Test
    "semiCirc", # Test
]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l

import re

def fix_column_name(col):
    # Ex: "MSE_ZZxReto_theta" -> "R2_ZZxReto_dtheta"
    match = re.match(r"^MSE_(.+)_([^_]+)$", col)
    if match:
        title, name = match.groups()
        return f"R2_{title}_d{name}"
    return col

results.columns = [fix_column_name(c) for c in results.columns]

In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZxReto_theta,R2_ZZxReto_dtheta,R2_ZZy1_theta,R2_ZZy1_dtheta,...,R2_LSG_2_theta,R2_LSG_2_dtheta,R2_ZZx1_inv_theta,R2_ZZx1_inv_dtheta,R2_ZZx1_theta,R2_ZZx1_dtheta,R2_ZZx2_inv_theta,R2_ZZx2_inv_dtheta,R2_semiCirc_theta,R2_semiCirc_dtheta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed3368,[1],0.3,0.7,0.01,3368,0.256553,0.360108,-1.627667,0.237465,...,-2.527434,0.092365,-0.192435,0.227165,0.512720,0.364132,-0.032188,0.170093,-4.841676,-0.100512
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed892,[1],0.3,0.7,0.01,892,0.734303,0.483577,-3.276844,0.270902,...,-1.954679,0.273054,0.533695,0.435531,0.339929,0.386944,-0.413988,0.087377,-8.560131,0.077568
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed2085,[1],0.3,0.7,0.01,2085,0.331454,0.380032,-1.970166,0.246551,...,-2.953922,0.100246,-0.176032,0.240217,0.500215,0.383990,-0.084414,0.171428,-5.576841,-0.097306
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed2613,[1],0.3,0.7,0.01,2613,0.377005,0.367732,-2.043142,0.236874,...,-3.117690,0.079809,-0.227239,0.217113,0.459549,0.373578,-0.063753,0.166505,-5.800866,-0.138212
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed3753,[1],0.3,0.7,0.01,3753,0.752595,0.579182,-8.079872,0.301091,...,-9.737026,0.209382,-0.935799,0.351651,0.054584,0.596485,-1.323965,0.002619,-20.652496,-0.168124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2746,model_arch100_r0.9_Ld0.7_Lp0.3_seed618,[100],0.7,0.3,0.90,618,-1.100234,0.515441,-15.413800,0.177789,...,-1.167290,0.400385,-2.222478,0.596674,-0.636499,0.386039,-2.409232,0.374133,-10.090642,0.232042
2747,model_arch100_r0.9_Ld0.7_Lp0.3_seed9474,[100],0.7,0.3,0.90,9474,-0.872498,0.469713,-14.574309,0.146979,...,-1.647476,0.369605,-1.909258,0.567554,-0.559823,0.339801,-1.929015,0.390331,-8.259140,0.259801
2748,model_arch100_r0.9_Ld0.7_Lp0.3_seed9889,[100],0.7,0.3,0.90,9889,-0.401432,0.498131,-15.159125,0.155528,...,-0.858177,0.414024,-0.992685,0.620048,-0.303953,0.372719,-2.125524,0.376385,-10.455175,0.222318
2749,model_arch100_r0.9_Ld0.7_Lp0.3_seed7658,[100],0.7,0.3,0.90,7658,-0.209439,0.528722,-16.664585,0.162818,...,-0.157116,0.441054,-0.488164,0.657396,-0.106634,0.416515,-2.391162,0.305029,-12.274820,0.188753


In [3]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZxReto":  "Train",
    "ZZy1":     "Train",
    "ZZx2":     "Val",
    "ZZy2":     "Val",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx1":     "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
595,model_arch29_r0.01_Ld0.7_Lp0.3_seed5131,[29],0.088292,-1.552414,-1.119387,-1.036248
572,model_arch28_r0.01_Ld0.7_Lp0.3_seed9143,[28],-0.209142,-1.895024,-0.862268,-1.144641
571,model_arch28_r0.01_Ld0.7_Lp0.3_seed5407,[28],-0.360524,-1.510851,-1.076784,-1.166183
656,model_arch32_r0.9_Ld0.7_Lp0.3_seed5407,[32],0.079810,-1.793939,-1.286742,-1.178847
908,model_arch45_r0.9_Ld0.3_Lp0.7_seed8756,[45],-0.167389,-1.080524,-1.626085,-1.179481
672,model_arch33_r0.01_Ld0.7_Lp0.3_seed9143,[33],0.221160,-1.856733,-1.369264,-1.182795
498,model_arch25_r0.9_Ld0.7_Lp0.3_seed9603,[25],0.284228,-2.062653,-1.266765,-1.192035
576,model_arch28_r0.9_Ld0.7_Lp0.3_seed5407,[28],-0.287348,-1.558179,-1.213752,-1.194989
717,model_arch35_r0.9_Ld0.7_Lp0.3_seed9143,[35],-0.145961,-1.764112,-1.200710,-1.211160
819,model_arch40_r0.9_Ld0.7_Lp0.3_seed868,[40],-0.386803,-1.213941,-1.403765,-1.212857



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZx2_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZx1_theta,R2_ZZx2_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
595,model_arch29_r0.01_Ld0.7_Lp0.3_seed5131,[29],0.629476,-0.452893,-0.305996,-2.798832,-0.710873,-1.294497,0.494516,0.082342,0.069791,-5.357600,0.088292,-1.552414,-1.119387,-1.036248
572,model_arch28_r0.01_Ld0.7_Lp0.3_seed9143,[28],0.617466,-1.035751,-0.834424,-2.955623,-0.957544,-0.637592,0.612965,0.256206,0.118907,-4.566550,-0.209142,-1.895024,-0.862268,-1.144641
571,model_arch28_r0.01_Ld0.7_Lp0.3_seed5407,[28],0.654641,-1.375689,-0.997296,-2.024405,-0.355784,-1.073815,0.604625,0.092753,0.182109,-5.910595,-0.360524,-1.510851,-1.076784,-1.166183
656,model_arch32_r0.9_Ld0.7_Lp0.3_seed5407,[32],0.586222,-0.426601,-0.854096,-2.733782,-0.489127,-1.879855,0.310338,-0.085794,0.064822,-5.640837,0.079810,-1.793939,-1.286742,-1.178847
908,model_arch45_r0.9_Ld0.3_Lp0.7_seed8756,[45],0.408891,-0.743669,-1.008276,-1.152772,-0.011506,-2.110921,0.283419,-0.537247,0.047370,-7.427627,-0.167389,-1.080524,-1.626085,-1.179481
672,model_arch33_r0.01_Ld0.7_Lp0.3_seed9143,[33],0.545800,-0.103481,-0.716210,-2.997255,-0.553639,-2.059865,0.244672,-0.138783,-0.081662,-5.626305,0.221160,-1.856733,-1.369264,-1.182795
498,model_arch25_r0.9_Ld0.7_Lp0.3_seed9603,[25],0.572221,-0.003766,-0.692021,-3.433285,-0.792385,-1.799444,0.288446,-0.002934,-0.058265,-5.236006,0.284228,-2.062653,-1.266765,-1.192035
576,model_arch28_r0.9_Ld0.7_Lp0.3_seed5407,[28],0.601885,-1.176582,-1.209023,-1.907334,-0.326382,-1.589848,0.429974,-0.132760,0.106464,-5.769958,-0.287348,-1.558179,-1.213752,-1.194989
717,model_arch35_r0.9_Ld0.7_Lp0.3_seed9143,[35],0.586624,-0.878545,-1.378114,-2.150111,-0.353029,-1.514450,0.417277,-0.106390,0.060079,-5.707748,-0.145961,-1.764112,-1.200710,-1.211160
819,model_arch40_r0.9_Ld0.7_Lp0.3_seed868,[40],0.552260,-1.325866,-1.059504,-1.368378,-0.077747,-1.485087,0.511950,-0.271272,-0.028665,-7.071768,-0.386803,-1.213941,-1.403765,-1.212857


In [4]:
final_table.to_excel("BestModels-1l.xlsx")